In [1]:
import chromadb

# 1. 連接到你的資料庫資料夾
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 2. 列出該資料夾下所有的 Collection 名稱
collections = client.list_collections()

print("--- 目前資料庫中的所有 Collection ---")
for col in collections:
    print(f"名稱: {col.name}")

--- 目前資料庫中的所有 Collection ---
名稱: pmi_specs
名稱: screw_manuals
名稱: hiwin_specs


In [4]:
import chromadb
from chromadb.utils import embedding_functions
import torch
# 1. 重新呼叫翻譯官 
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  # 確保調用你的 RTX 4060
)
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
# 2. 連接到硬碟裡的資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 3. 取得當初存入 Chunks 的 Collection
manual_collection = client.get_collection(name = "screw_manuals", embedding_function = emb_fn)
spec_collection = client.get_collection(name = "hiwin_specs", embedding_function = emb_fn)


print("成功連接向量資料庫，Chunks 已準備好被檢索！")

CUDA 是否可用: True
成功連接向量資料庫，Chunks 已準備好被檢索！


In [5]:
import ollama

# 定義專業知識字典
SERIES_INFO = {
    "FDC": "雙螺帽設計，具備極高的軸向剛性與預壓穩定性，專為重負荷精密工具機設計。",
    "FSW": "小法蘭單螺帽設計，體積精簡，適合安裝空間受限的自動化設備。",
    "FSV": "標準單螺帽型，具備優異的傳動效率與流暢度，是自動化產業最泛用的標準件。",
    "RSI": "旋轉螺帽設計，適合絲槓固定、螺帽旋轉的機構，能有效抑制長行程下的振動。",
    "FSI": "內循環設計，螺帽外徑小，運轉安靜，適合小型精密設備。"
}

def get_expert_advice(user_query, calc_result, use_rag=True):
    """
    混合檢索架構：同時檢索技術手冊 (Manual) 與 產品規格 (Specs)
    """
    rag_context = ""
    rag_status_msg = ""

    # --- RAG 混合檢索邏輯 ---
    if use_rag:
        try:
            # A. 檢索【技術手冊】(Manual Chunks): 找潤滑、安裝、原理、壽命
            manual_res = manual_collection.query(query_texts=[user_query], n_results=2)
            manual_text = "\n【技術手冊參考資料】：\n" + "\n".join(manual_res['documents'][0])
            
            # B. 檢索【產品規格】(Specs Documents): 找替代型號、詳細尺寸、參數對比
            spec_res = spec_collection.query(query_texts=[user_query], n_results=3)
            spec_text = "\n【相似型號規格參考】：\n" + "\n".join(spec_res['documents'][0])
            
            # 整合兩路檢索結果
            rag_context = f"{spec_text}\n{manual_text}"
            rag_status_msg = "\n(系統提示：已完成混合檢索 - 參考手冊與規格表)\n"
            
        except Exception as e:
            rag_context = f"\n(系統提示：資料庫檢索失敗: {e})\n"
    
    # --- 數據準備 (目前計算出的最優解) ---
    series = calc_result.get('series', '標準')
    model = calc_result.get('model', '未知')
    feature = SERIES_INFO.get(series, "HIWIN 精密傳動元件。")
    
    spec_context = f"""
    【目前推薦型號數據】
    - 推薦系列：{series} ({feature})
    - 具體型號：{model}
    - 物理參數：公稱外徑 {calc_result['dia']}mm, 導程 {calc_result['lead']}mm
    - 動負荷能力：{calc_result['dynamic_load']} kgf
    """
    
    # --- 組合最終 Prompt ---
    # 這裡我們明確區分「目前數據」與「參考資料」，幫助 LLM 進行對比分析
    prompt = f"""
    你是一位專業的 HIWIN 技術支援工程師，請根據提供的【目前推薦型號數據】與【參考資料】來回答提問。
    回答時請結合產品的物理特性（如外徑、負荷）與系列優點（如剛性、空間利用)，請用繁體中文回答。
       
    當使用者詢問關於空間、尺寸或替代型號時，請優先參考「相似型號規格」進行對比。
    當使用者詢問關於安裝、保養或技術原理時，請參考「技術手冊參考資料」。

    {spec_context}
    
    {rag_context}
    
    使用者提問：{user_query}
    
    請用繁體中文回答，語氣專業且誠懇，並儘可能引用具體參數。
    """
    
    # --- 呼叫 Qwen 2.5 ---
    try:
        response = ollama.generate(
            model='qwen2.5:7b', 
            prompt=prompt,
            options={"temperature": 0.3} 
        )
        return response['response'] + rag_status_msg
    except Exception as e:
        return f"連線 Ollama 發生錯誤: {str(e)}"

In [23]:
#模型:Ollama qwen2.5:7b，無RAG回答

# 模擬你公式跑完後的結果 (Dictionary 格式)
calc_result = {
    "series": "FDC",
    "model": "40-12K5",
    "dia": 40.0,
    "lead": 12.0,
    "dynamic_load": 7430
}

# 測試提問
user_query = "螺帽直徑跟長度空間有限，是否有其他型號建議"

print("正在調用 Qwen 2.5:7b 進行分析...\n")
result = get_expert_advice(user_query, calc_result, use_rag = False)

from IPython.display import Markdown
Markdown(result) # 使用 Markdown 讓回答看起來更漂亮

正在調用 Qwen 2.5:7b 進行分析...



您好，

感謝您對HIWIN產品的興趣。關於您的問題，根據目前推薦的FDC系列螺帽設計及其物理特性，我們可以提供以下建議：

1. **型號40-12K5**：此型號具有公稱外徑 40.0mm 和導程 12.0mm 的特點。其軸向剛性與預壓穩定性極高，適合重負荷精密工具機使用。動負荷能力為7430 kgf。

若考慮螺帽直徑和長度空間有限的問題，我們可以參考相似型號規格進行對比：

- **型號40-10K5**：此型號與40-12K5相比，導程較短（10.0mm），外徑相同。雖然軸向剛性與預壓穩定性可能會稍有下降，但仍然適合中等負荷應用。

- **型號36-12K5**：此型號的公稱外徑減小為36.0mm，導程仍保持在12.0mm。這可能更符合您提到的空間限制需求，同時仍能提供良好的軸向剛性和預壓穩定性。

以上建議皆基於相似型號規格進行比較，具體選擇需根據您的負荷需求和實際安裝空間來決定。我們建議在確定最終選型前，進一步確認您的具體應用條件。

如有更多技術問題或需要更詳細的資料，歡迎隨時聯繫我們。感謝您的信任與支持！

敬上

In [6]:
#模型:Ollama qwen2.5:7b，有RAG回答

# 模擬你公式跑完後的結果 (Dictionary 格式)
calc_result = {
    "series": "FDC",
    "model": "40-12K5",
    "dia": 40.0,
    "lead": 12.0,
    "dynamic_load": 7430
}

# 測試提問
user_query = "螺帽直徑跟長度空間有限，是否有其他型號建議?"

print("正在調用 Qwen 2.5:7b 進行分析...\n")
result = get_expert_advice(user_query, calc_result, use_rag = True)

from IPython.display import Markdown
Markdown(result) # 使用 Markdown 讓回答看起來更漂亮

正在調用 Qwen 2.5:7b 進行分析...



根據您的需求，考慮到螺帽直徑和長度空間的限制，我們可以推薦以下幾種替代型號：

1. **HIWIN 32-8T4 (系列: RSI)**：此型號具有公稱外徑 32mm、導程 8.0mm 的特點。其節圓直徑為 33.0mm，根徑為 28.132mm，珠卷數為 4。雖然在公稱外徑上較小於您目前推薦的 FDC 40-12K5 型號，但其軸向剛性達 47 kfg/umk，額定動負荷(C)為 2317 kgf，靜負荷(Co)為 5635 kgf。此型號適合長行程且需高速旋轉螺帽的特殊機構。

2. **HIWIN 45-10B1 (系列: FSW)**：此型號具有公稱外徑 45mm、導程 10.0mm 的特點，節圓直徑為 46.4mm，根徑為 39.91mm，珠卷數為 2.5x1。其軸向剛性達 45 kfg/umk，額定動負荷(C)為 3116 kgf，靜負荷(Co)為 7953 kgf。此型號結構輕巧省空間，適合小型自動化設備或精密儀器。

兩種替代型號皆具有不同的物理特性與優點：
- **HIWIN 32-8T4**：在軸向剛性方面表現較佳，且動負荷能力也相對較大。
- **HIWIN 45-10B1**：結構更為輕巧，適合空間有限的應用場景。

建議您可以根據具體的工作環境和需求來選擇合適的型號。如果您需要進一步詳細的信息或技術支援，歡迎隨時聯繫我們。
(系統提示：已完成混合檢索 - 參考手冊與規格表)
